# QA 07 — model parameters and hyperparameters

Estimator-owned regularization, optimization, weighting, topology, activation, loss, and task variants.

> Run this notebook from top to bottom after installing the project with
> `pip install -e ".[notebooks]"`. Figures are genuine public-API outputs.
> Cells intentionally contain no assertions: automated invariants live in
> `tests/`, while this notebook is for human visual inspection.

In [ ]:
from pathlib import Path
import sys

candidate = Path.cwd().resolve()
while candidate != candidate.parent and not (candidate / "pyproject.toml").exists():
    candidate = candidate.parent
if not (candidate / "pyproject.toml").exists():
    raise RuntimeError("Open this notebook from inside the Mlektic repository.")
ROOT = candidate
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
from IPython.display import display
from sklearn.linear_model import LogisticRegression, SGDRegressor
from mlektic import TorchTrainingRecorder, visualize_lr, visualize_logistic, visualize_nn_architecture, visualize_nn_training
from notebooks._support import binary_case, case_heading, linear_case, multiclass_case

### `HYPER-LR-L2`

**Inspect:** SGDRegressor L2 regularization strength exposed in the complete panel

In [ ]:
case_heading("HYPER-LR-L2", "SGDRegressor L2 regularization strength exposed in the complete panel")
c=linear_case(2)
m=SGDRegressor(penalty='l2',alpha=0.001,max_iter=500,random_state=17).fit(c.X,c.y)
display(visualize_lr(m,c.X,c.y,steps=16,max_frames=7,detail='complete'))

### `HYPER-LR-L1`

**Inspect:** SGDRegressor L1 penalty and sparse-coefficient tendency

In [ ]:
case_heading("HYPER-LR-L1", "SGDRegressor L1 penalty and sparse-coefficient tendency")
c=linear_case(6)
m=SGDRegressor(penalty='l1',alpha=0.02,max_iter=800,random_state=17).fit(c.X,c.y)
display(visualize_lr(m,c.X,c.y,steps=16,max_frames=7,detail='complete',size='wide'))

### `HYPER-LR-LEARNING-RATE`

**Inspect:** constant learning-rate replay metadata without claiming original fit recovery

In [ ]:
case_heading("HYPER-LR-LEARNING-RATE", "constant learning-rate replay metadata without claiming original fit recovery")
c=linear_case(1)
m=SGDRegressor(learning_rate='constant',eta0=0.01,max_iter=300,random_state=17).fit(c.X,c.y)
display(visualize_lr(m,c.X,c.y,steps=18,max_frames=8,detail='complete'))

### `HYPER-LOG-C-STRONG`

**Inspect:** stronger logistic L2 regularization through a smaller C

In [ ]:
case_heading("HYPER-LOG-C-STRONG", "stronger logistic L2 regularization through a smaller C")
c=binary_case(2)
m=LogisticRegression(C=0.1,max_iter=1000,random_state=17).fit(c.X,c.y)
display(visualize_logistic(m,c.X,c.y,steps=12,max_frames=6,detail='complete',show_loss=True))

### `HYPER-LOG-C-WEAK`

**Inspect:** weaker logistic L2 regularization through a larger C

In [ ]:
case_heading("HYPER-LOG-C-WEAK", "weaker logistic L2 regularization through a larger C")
c=binary_case(2)
m=LogisticRegression(C=100.0,max_iter=1000,random_state=17).fit(c.X,c.y)
display(visualize_logistic(m,c.X,c.y,steps=12,max_frames=6,detail='complete',show_loss=True))

### `HYPER-LOG-L1`

**Inspect:** binary L1 penalty with a compatible public solver

In [ ]:
case_heading("HYPER-LOG-L1", "binary L1 penalty with a compatible public solver")
c=binary_case(6)
m=LogisticRegression(penalty='l1',solver='liblinear',C=0.5,max_iter=1000,random_state=17).fit(c.X,c.y)
display(visualize_logistic(m,c.X,c.y,steps=12,max_frames=6,detail='complete',size='wide'))

### `HYPER-LOG-CLASS-WEIGHT`

**Inspect:** balanced class weights on an imbalanced dataset

In [ ]:
case_heading("HYPER-LOG-CLASS-WEIGHT", "balanced class weights on an imbalanced dataset")
c=binary_case(2,imbalanced=True)
m=LogisticRegression(class_weight='balanced',max_iter=1000,random_state=17).fit(c.X,c.y)
display(visualize_logistic(m,c.X,c.y,steps=12,max_frames=6,detail='complete',show_loss=True))

### `HYPER-LOG-MULTICLASS-C`

**Inspect:** multiclass regularization and complete matrix mathematics

In [ ]:
case_heading("HYPER-LOG-MULTICLASS-C", "multiclass regularization and complete matrix mathematics")
c=multiclass_case(3,classes=4)
m=LogisticRegression(C=0.4,max_iter=1200,random_state=17).fit(c.X,c.y)
display(visualize_logistic(m,c.X,c.y,steps=10,max_frames=6,detail='complete',show_loss=True))

### `HYPER-NN-DEEP`

**Inspect:** deeper dense architecture with BatchNorm, ReLU, and Dropout roles

In [ ]:
case_heading("HYPER-NN-DEEP", "deeper dense architecture with BatchNorm, ReLU, and Dropout roles")
import torch
torch.manual_seed(17)
m=torch.nn.Sequential(torch.nn.Linear(6,12),torch.nn.BatchNorm1d(12),torch.nn.ReLU(),torch.nn.Dropout(0.2),torch.nn.Linear(12,4),torch.nn.Tanh(),torch.nn.Linear(4,1))
m.eval()
display(visualize_nn_architecture(m,torch.zeros(4,6),max_layers=10,theme='academic',size='wide'))

### `HYPER-NN-CONV`

**Inspect:** convolution, activation, flattening, and dense output dimensions

In [ ]:
case_heading("HYPER-NN-CONV", "convolution, activation, flattening, and dense output dimensions")
import torch
torch.manual_seed(17)
m=torch.nn.Sequential(torch.nn.Conv2d(1,3,kernel_size=3),torch.nn.ReLU(),torch.nn.Flatten(),torch.nn.Linear(108,2))
display(visualize_nn_architecture(m,torch.zeros(1,1,8,8),max_layers=8,theme='classroom',size='wide'))

### `HYPER-NN-REGRESSION`

**Inspect:** recorded MSE training for a neural regression task

In [ ]:
case_heading("HYPER-NN-REGRESSION", "recorded MSE training for a neural regression task")
import torch
torch.manual_seed(17)
X=torch.linspace(-1,1,20).reshape(-1,1); y=1+2*X
m=torch.nn.Sequential(torch.nn.Linear(1,6),torch.nn.Tanh(),torch.nn.Linear(6,1))
opt=torch.optim.Adam(m.parameters(),lr=0.05); loss_fn=torch.nn.MSELoss(); rec=TorchTrainingRecorder(m,optimizer=opt,loss_fn=loss_fn)
for step in range(10):
 opt.zero_grad(); pred=m(X); loss=loss_fn(pred,y); loss.backward(); opt.step(); rec.record(step+1,loss=loss,predictions=pred,targets=y,task='regression')
rec.close(); display(visualize_nn_training(rec.to_history(),max_frames=8,theme='academic'))

### `HYPER-NN-MULTICLASS`

**Inspect:** recorded CrossEntropy training and inferred multiclass metrics

In [ ]:
case_heading("HYPER-NN-MULTICLASS", "recorded CrossEntropy training and inferred multiclass metrics")
import torch
torch.manual_seed(17)
X=torch.randn(24,3); y=torch.arange(24)%3
m=torch.nn.Sequential(torch.nn.Linear(3,8),torch.nn.ReLU(),torch.nn.Linear(8,3))
opt=torch.optim.SGD(m.parameters(),lr=0.1); loss_fn=torch.nn.CrossEntropyLoss(); rec=TorchTrainingRecorder(m,optimizer=opt,loss_fn=loss_fn)
for step in range(8):
 opt.zero_grad(); pred=m(X); loss=loss_fn(pred,y); loss.backward(); opt.step(); rec.record(step+1,loss=loss,predictions=pred,targets=y,task='classification')
rec.close(); display(visualize_nn_training(rec.to_history(),max_frames=8,theme='classroom'))